# Zombie modelling

TO DO:
* ~Fix SIR model equations to account for fixed population~
* Work out map projection so it looks normal
* Calculate population over time graphs more efficiently by applying a single mask over entire array
* Include diagonal interaction terms 
* Experiment with beta and gamma parameters to reduce required computation time


In [ ]:
# Import dependencies

import os
import glob
import matplotlib.pyplot as plt
import numpy as np

import rasterio
from rasterio.plot import show
import imageio

from tqdm import tqdm

import rioxarray

## Format population density map
The data is a map of the population density of a given country in 2020, at a resolution of approximately 1km^2 per pixel. 

In [ ]:
# The population density map is at the following location
file = 'Data/PopDensity/Germany/2020/deu_pd_2020_1km_UNadj.tif'
#file = 'Data/PopDensity/Czechia/2020/cze_pd_2020_1km_UNadj.tif'
country = 'Germany'
#country = 'Czechia'

with rasterio.open(file) as src:
    full_img = src.read()
    band = src.read(1)
    meta = src.meta
    transform = src.transform
    crs = src.crs
    mask = np.array(src.read_masks(1)/255, dtype=bool)
    
# Create a copy of the band with the masked values represented with 0 instead of -99999.0
band0 = np.where(band <0, 0, band)

In [ ]:
print(f'The shape of the image is {band.shape}')
print(f'The shape of the mask is {mask.shape}')
print(f'The mask is represented in the image by pixels with a value of {meta["nodata"]}')
aspect_ratio = band.shape[1]/band.shape[0]
print(f'The aspect ratio is {aspect_ratio}')
print(f'The coordinate reference system (CRS) is {crs}')

In [ ]:
# The mask is an array where the pixels for which we have data are white (True), and the pixels for which we do not have data are black (False)
show(mask,transform=transform, title='Mask', cmap='Greys_r')

plt.show()

In [ ]:
# We can apply the mask to our data to obtain only the parts for which we have data (we need to invert the mask because we are 'masking away' the bad stuff
masked_band = np.ma.masked_array(band, ~mask)

In [ ]:
# Let's visualise this data now

cmap = plt.cm.get_cmap("Blues_r").copy()
cmap.set_bad(color='grey')


fig, ax = plt.subplots(figsize=(8,8))
show(masked_band, transform=transform, title=f'Population density in {country}', cmap=cmap)
plt.show()



In [ ]:
# Zooming in on the highest population density, we see
fig, ax = plt.subplots(figsize=(8,8))

# Finds the index of the highest population density
max_idx = np.unravel_index(masked_band.argmax(), masked_band.shape)

# Plots the region around the area with the highest population density
show(masked_band[int(np.floor(max_idx[0] - 0.1*band.shape[0])):int(np.floor(max_idx[0] + 0.1*band.shape[0])), int(np.floor(max_idx[1] - 0.1*band.shape[1])): int(np.floor(max_idx[1] + 0.1*band.shape[1]))], transform=transform,  title=f'Region of highest population density in {country}', cmap=cmap)
plt.show()

In [ ]:
# By taking the mean of the masked_band array, we get the population density across germany. Multiplying this by area should net the total population.
mean_pop_dens = masked_band.mean() #in people per square kilometer
total_area = (~mask).sum()  # in square kilometers (average only)

total_population = mean_pop_dens*total_area 
print(f'The population of {country} is approximately {total_population/1000000:.1f} million people')



## SIR Model (with migration)
The model we are implementing is a slight modification on the usual SIR model consisting of a local SIR model (i.e. an SIR model at each pixel) together with a migration term (movement to nearby pixels). This models a population that can only infect locally, but can also move.

## The usual SIR model
The SIR model is the simplest mathematical model to describe disease spread, and involves dividing the population into three separate groups:
* Susceptible ($S$) - The healthy members of the population
* Infected ($I$) - The zombie members of the population
* Removed ($R$) - The zombies who have been killed


At any given time the zombies have the potential to infect members of the healthy group. The details of how the disease spreads is a complicated topic, but on average we can describe the rate of transmission with a single number, the infection rate parameter $\beta$. Bigger $\beta$ means a more infectious disease. The number of new infected at each step is proportional to this parameter $\beta$, the number of current zombies (more zombies = more bites), as well as the current number of healthy individuals nearby (high population density means diseases spread more quickly). Similarly, the rate of removal is described with a single number, $\gamma$, which describes the rate at which zombies are killed (we assume there is no cure). 

## How does the population density map come into this?
We will use the population density map obtained at the start of this notebook as our initial values for the susceptible population. Each pixel is a gridpoint, and the brightness of that pixel represents the population density, and actually the total population at each pixel. We will assume that almost all of the population is initially uninfected (and therefore susceptible), and numerically solve the above equations when we introduce a small infected population into the most densely populated part of the country. 

We model the Susceptible, Infected, and Removed populations as densities, with migration terms being modelled with diffusion terms.The equations are

$\frac{dS}{dt} = -\frac{\beta IS}{S + I + R} + \mu_S \Delta S$

$\frac{dI}{dt} = \frac{\beta IS}{S + I + R} - \gamma I + \mu_I \Delta I $

$\frac{dR}{dt} = \gamma I + \mu_R \Delta R$

In [ ]:
#Define beta and gamma, as well as the migration constants mu
beta = 0.3
gamma = 0.01

mu_S = 0
mu_I = 0.1
mu_R = 0


By rearranging the definition of the derivative, we have

$u_{n+1} = f(u) \Delta t + u_n$

where $u'(t) = f(u)$

In [ ]:
# This step calculates the next step from the previous one, we just need to implement u and f
# We also clip the values here so they don't go above the maximum or below the minimum
def euler_step(u, f, dt):
    newu = u + dt * f(u)
    
    return newu

In [ ]:
# Defines the laplacian of a function g
def laplace(g):
    '''
    Takes in a 2D numpy array and returns the discrete laplacian on that array
    '''
    result = g[0:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, 0:-2] + g[1:-1, 2:] - 4*g[1:-1, 1:-1]
    
    return result

# f, the function which controls the RHS of the derivative, will be given by
def f(u):
    S = u[0]
    I = u[1]
    R = u[2]
    
    
    new = np.array([np.divide(-beta*S[1:-1, 1:-1]*I[1:-1, 1:-1], S[1:-1, 1:-1] + I[1:-1, 1:-1] + R[1:-1, 1:-1], \
                              where=S[1:-1, 1:-1] + I[1:-1, 1:-1] + R[1:-1, 1:-1]!=0) + mu_S*laplace(S),
                    np.divide(beta*S[1:-1, 1:-1]*I[1:-1, 1:-1], S[1:-1, 1:-1] + I[1:-1, 1:-1] + R[1:-1, 1:-1], \
                              where=S[1:-1, 1:-1] + I[1:-1, 1:-1] + R[1:-1, 1:-1]!=0) - gamma*I[1:-1, 1:-1] + mu_I*laplace(I),
                     gamma*I[1:-1, 1:-1] + mu_R*laplace(R)
                    ])
    
    # Here we want to apply the mask, and clip any populations that go below zero to be zero
    # We also want to clip any populations which go above the population in that pixel to the max
    padding = np.zeros_like(u)
    padding[:,1:-1,1:-1] = new
    
    # First apply the mask to each layer
    padding[0] = np.ma.masked_array(padding[0], ~mask)
    padding[1] = np.ma.masked_array(padding[1], ~mask)
    padding[2] = np.ma.masked_array(padding[2], ~mask)
    

    
    return padding

In [ ]:
# Set the number of timesteps you want
T = 300                          # final time
dt = 1                          # time increment
N = int(T/dt) + 1               # number of time-steps
t = np.linspace(0.0, T, N)      # time discretization

# u will be a multi-dimensional array consisting of the S,I, and R values at each gridpoint and each time
u = np.zeros((N, 3, band.shape[0], band.shape[1]))

$ S_{i,j}(t) = u[t, 0, i, j]$

$ I_{i,j}(t) =  u[t, 1, i, j]$

$ R_{i,j}(t) =  u[t, 2, i, j]$

In [ ]:
# The initial susceptible population is just the population density
u[0, 0, :,:] = masked_band

# We need to set the masked data to zero (masked data was originally -9999)
u[0,0][u[0,0]<0] = 0

# The initial infected population is mostly zero
# However we will inject a few infected patients at the most densely populated part of the country. Let's say 1% of the densest 1km^2
u[0, 0, max_idx[0], max_idx[1]] = 0.99*masked_band[max_idx[0], max_idx[1]] 
u[0, 1, max_idx[0], max_idx[1]] = 0.01*masked_band[max_idx[0], max_idx[1]] 


# The removed part of the population is initially zero

In [ ]:
# Run the code
for n in tqdm(range(N-1)):
    u[n+1] = euler_step(u[n], f, dt)

In [ ]:
uRGB = np.transpose(u, (0,2,3,1)).copy()
# We want the infected to be red, the healthy to be green, and the dead to be blue
uRGB[:,:,:,[1,0]] = uRGB[:,:,:,[0,1]]


# Rescale all elements so they are between 0 and 1
uRGB = uRGB/uRGB.max()



In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
plt.imshow(np.clip(uRGB[0, :, :, :]*40,0,1),    # display first slice              
                    vmin=0, # lowest value in numpy_3d_array
                    vmax=1)
plt.savefig('1.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
plt.imshow(np.clip(uRGB[N-1, :, :, :]*40,0,1),    # display first slice              
                    vmin=0, # lowest value in numpy_3d_array
                    vmax=1)

plt.savefig('2.png')
plt.show()

In [ ]:
health_pop = [np.ma.masked_array(u[t,0], ~mask).mean()*total_area/1000000 for t in range(N-1)]
zombie_pop = [np.ma.masked_array(u[t,1], ~mask).mean()*total_area/1000000 for t in range(N-1)]
dead_pop = [np.ma.masked_array(u[t,2], ~mask).mean()*total_area/1000000 for t in range(N-1)]

In [ ]:
plt.plot(health_pop)
plt.plot(zombie_pop)
plt.plot(dead_pop)

plt.show()

In [ ]:
# Zooming in on the highest population density, we see
fig, ax = plt.subplots(figsize=(8,8))

# Finds the index of the highest population density
max_idx = np.unravel_index(masked_band.argmax(), masked_band.shape)

# Plots the region around the area with the highest population density
show(u[0,0][int(np.floor(max_idx[0] - 0.1*band.shape[0])):int(np.floor(max_idx[0] + 0.1*band.shape[0])), int(np.floor(max_idx[1] - 0.1*band.shape[1])): int(np.floor(max_idx[1] + 0.1*band.shape[1]))], transform=transform,  title=f'Region of highest population density in {country} before the infection', cmap="inferno")
plt.show()

In [ ]:
# Zooming in on the highest population density, we see
fig, ax = plt.subplots(figsize=(8,8))

# Finds the index of the highest population density
max_idx = np.unravel_index(masked_band.argmax(), masked_band.shape)

# Plots the region around the area with the highest population density
show(u[N-1,0][int(np.floor(max_idx[0] - 0.1*band.shape[0])):int(np.floor(max_idx[0] + 0.1*band.shape[0])), int(np.floor(max_idx[1] - 0.1*band.shape[1])): int(np.floor(max_idx[1] + 0.1*band.shape[1]))], transform=transform,  title=f'Region of highest population density in {country} after the infection', cmap="inferno")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
plt.imshow(np.clip(uRGB[0, :, :, :]*40,0,1),    # display first slice              
                    vmin=0, # lowest value in numpy_3d_array
                    vmax=1)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import animation


# generating gif out of slices of 4-dimensional numpy array
def generate_gif(numpy_array):
    
#    fig = plt.subplots(figsize=(10,10)
    fig = plt.figure()
    im = plt.imshow(np.clip(numpy_array[0, :, :, :]*40,0,1),    # display first slice      
                    animated=True,
                    vmin=0, # lowest value in numpy_3d_array
                    vmax=1)

#    im = plt.imshow(numpy_array[0, :, :, :],    # display first slice
#                    animated=True,
#                    cmap='Reds_r',               # color mapping
#                    vmin=np.iinfo('uint8').min, # lowest value in numpy_3d_array
#                    vmax=np.iinfo('uint8').max) # highest value in numpy_3d_array
#    if channel==0:
#        plt.colorbar(label='Healthy', shrink=0.75)
#    elif channel==1:
#        plt.colorbar(label='Infected', shrink=0.75)
#    else:
#        plt.colorbar(label='Dead', shrink=0.75)

        
    
    plt.tight_layout()

    def init():
        im.set_data(np.clip(numpy_array[0, :, :, :]*40,0,1))
        return im,

    def animate(i):
        im.set_array(np.clip(numpy_array[i, :, :, :]*40,0,1))
        return im,

    # calling animation function of matplotlib
    anim = animation.FuncAnimation(fig,
                                   animate,
                                   init_func=init,
                                   frames=range(0, np.shape(numpy_array)[0], 5),  # amount of frames being animated
                                   interval=1,                       # Delay between frames in milliseconds.
                                   blit=True,
                                   repeat_delay=1000)        #How long to wait before restarting the gif in milliseconds
    anim.save("RGB.gif")   # save as gif
    plt.show()
generate_gif(uRGB)